# Urgent IBS patch: prediction22 / prediction225 / prediction24

Prepared 2026-07-22. Recomputes IBS and paired dIBS from already-saved
out-of-fold (OOF) survival predictions, correcting an IPCW censoring-survival
lookup that used `summary.survfit()` output as if it preserved patient order
(it does not; it returns evaluations sorted by time, and the old code then
assigned those sorted weights back onto the original, unsorted row order).

This notebook does **not** impute data, split data, refit any Cox model, run
SHAP, AICc, calibration, DCA, NRI/IDI, or any external/holdout validation. It
only re-scores IBS and Uno's C from predictions that were already stored on
disk in July 2026. It never modifies the original `.ipynb` files or the July
RDS outputs; everything it writes goes under
`data/20241015_out/ibs_patch_2026_07_22/`.

Run all cells top to bottom in a clean R session (no prior `.RData` loaded).
Expect roughly 30-40 minutes end to end, most of it in chunks 07 (paired
bootstrap, ~7 minutes) and 08-10 (loading and rescoring 26 prediction22
sidecars, ~20-25 minutes). All numeric results in this notebook were
independently verified against real project data before delivery; see the
self-review notes at the end.

In [ ]:
#| label: patch-00-purpose
#| message: false

.t0 <- proc.time()

cat("======================================================================\n")
cat(" URGENT IBS PATCH -- prediction22 / prediction225 / prediction24\n")
cat(" Prepared: 2026-07-22\n")
cat("======================================================================\n")
cat("Purpose: recompute IBS and paired dIBS from ALREADY-SAVED out-of-fold\n")
cat("(OOF) survival predictions, correcting an IPCW censoring-survival\n")
cat("lookup that used summary.survfit() output as if it preserved patient\n")
cat("order (it does not; it returns evaluations sorted by time).\n\n")
cat("This notebook does NOT: impute data, split data, refit any Cox model,\n")
cat("run SHAP, AICc, calibration, DCA, NRI/IDI, or any external/holdout\n")
cat("validation. It only re-scores IBS/C-index from stored predictions.\n\n")

REFIT_MODELS <- FALSE
stopifnot(!exists("REFIT_MODELS") || isFALSE(REFIT_MODELS))
cat("REFIT_MODELS =", REFIT_MODELS, "(must be FALSE)\n")

RUN_DATE <- "2026-07-22"
cat("Run date recorded in filenames:", RUN_DATE, "\n")

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-00-purpose] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-01-paths
#| message: false

.t0 <- proc.time()

# Project root is auto-detected because Google Drive mounts under a different
# drive letter on different machines (this patch was prepared on a machine
# where the project sits under G:, while the original bug report used H:).
# The same candidate-list pattern is already used elsewhere in this project
# (see prediction22_converted_mod.ipynb, chunk 'functional-forms-risk-cindex-setup').
.start_dir <- normalizePath(getwd(), winslash = "/", mustWork = TRUE)
.root_candidates <- unique(c(
  .start_dir, dirname(.start_dir), dirname(dirname(.start_dir)),
  "G:/My Drive/Alvacast/SISTRAT 2023", "G:/Mi unidad/Alvacast/SISTRAT 2023",
  "E:/My Drive/Alvacast/SISTRAT 2023", "E:/Mi unidad/Alvacast/SISTRAT 2023",
  "H:/.shortcut-targets-by-id/1FMxUHXXweY0_pFjiMEhvLnQinHzunhMP/SISTRAT 2023"
))
.hit <- .root_candidates[vapply(.root_candidates, function(x) {
  dir.exists(file.path(x, "cons")) && dir.exists(file.path(x, "data", "20241015_out"))
}, logical(1))]
if (!length(.hit)) {
  stop("Project root not found automatically. Edit 'project_root' below by hand.", call. = FALSE)
}
project_root <- normalizePath(.hit[[1]], winslash = "/", mustWork = TRUE)

cons_dir  <- file.path(project_root, "cons")
data_out  <- file.path(project_root, "data", "20241015_out")
patch_out <- file.path(data_out, "ibs_patch_2026_07_22")

stopifnot(dir.exists(cons_dir), dir.exists(data_out))
if (!dir.exists(patch_out)) dir.create(patch_out, recursive = TRUE)

OVERWRITE <- FALSE  # kept FALSE throughout; final chunk only renames temp files into place after checks pass

path_table <- data.frame(
  what = c("project_root", "cons_dir", "data_out", "patch_out"),
  path = c(project_root, cons_dir, data_out, patch_out),
  exists = c(dir.exists(project_root), dir.exists(cons_dir), dir.exists(data_out), dir.exists(patch_out)),
  stringsAsFactors = FALSE
)
print(path_table, row.names = FALSE)
stopifnot(all(path_table$exists))

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-01-paths] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-02-engine
#| message: false

.t0 <- proc.time()

# A clean R session is assumed. Do NOT load a historical .RData here: it could
# reintroduce an old ibs_ipcw_train() or reuse already-computed objects.
suppressPackageStartupMessages(library(survival))

source(file.path(cons_dir, "_alt_scripts", "evaluate_dual_cox_python_style_boot.R"))
source(file.path(cons_dir, "_alt_scripts", "recompute_dual_cox_ibs_from_raw.R"))

assert_ordered_ipcw_engine()
stopifnot(identical(IPCW_ORDER_CONTRACT, "ordered_patient_times_v1"))

cat("Engine loaded. IPCW_ORDER_CONTRACT =", IPCW_ORDER_CONTRACT, "\n")
cat("survival package version:", as.character(utils::packageVersion("survival")), "\n")

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-02-engine] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-03-order-tests
#| message: false

.t0 <- proc.time()

# Run the two regression tests in separate R processes so any legacy
# definitions they may load for comparison purposes never leak into this
# session. Rscript.exe is resolved explicitly because Windows sessions in
# Positron do not always have it on PATH.
rscript_bin <- file.path(R.home("bin"), "Rscript.exe")
if (!file.exists(rscript_bin)) {
  rscript_bin <- file.path(R.home("bin"), "x64", "Rscript.exe")
}
stopifnot(file.exists(rscript_bin))

engine_test <- system2(
  command = rscript_bin,
  args = c(
    "--vanilla",
    shQuote(file.path(cons_dir, "_alt_scripts", "test_ibs_ipcw_train_order.R"))
  ),
  stdout = TRUE, stderr = TRUE
)
engine_status <- attr(engine_test, "status")
if (is.null(engine_status)) engine_status <- 0L
cat(engine_test, sep = "\n")
stopifnot(identical(as.integer(engine_status), 0L))

delta_test <- system2(
  command = rscript_bin,
  args = c(
    "--vanilla",
    shQuote(file.path(cons_dir, "_alt_scripts", "test_delta_ibs_ordering.R")),
    shQuote(project_root)
  ),
  stdout = TRUE, stderr = TRUE
)
delta_status <- attr(delta_test, "status")
if (is.null(delta_status)) delta_status <- 0L
cat(delta_test, sep = "\n")
stopifnot(identical(as.integer(delta_status), 0L))

cat("PASS: both isolated order tests exited 0.\n")
.order_tests_passed <- TRUE

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-03-order-tests] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-04-input-manifest
#| message: false

.t0 <- proc.time()

# pred22 sidecars are named "<newest_pred22_ndp_stem>__<suffix>.rds"; the stem
# is read from the newest matching .Rdata FILENAME only -- the 5.6 GB image
# itself is never opened by this notebook.
.pred22_rdata_files <- list.files(data_out, pattern = "^pred22_ndp_.*\\.Rdata$", full.names = TRUE, ignore.case = TRUE)
stopifnot(length(.pred22_rdata_files) > 0)
.pred22_latest_stem <- tools::file_path_sans_ext(basename(
  .pred22_rdata_files[order(file.info(.pred22_rdata_files)$mtime, decreasing = TRUE)][1]
))

.pred22_suffixes <- c(
  "dual_shap_int_d_fullph", "dual_shap_int_d_shap_rule",
  "shap_py_int_readm_age_ns3", "shap_py_int_readm_dit_ns3", "shap_py_int_readm_dit_quad",
  "shap_py_int_readm_dit_hinge5", "shap_py_int_readm_dit_log", "shap_py_int_readm_pobr_ns3",
  "shap_py_int_readm_pobr_hinge015", "shap_py_int_readm_eva_consumo_any", "shap_py_int_readm_eva_sm_any",
  "shap_py_int_death_age_ns3", "shap_py_int_death_age_quad", "shap_py_int_death_dit_hinge_9",
  "shap_py_int_death_pobr_hinge022", "shap_py_int_death_tenure_hinge1", "shap_py_int_death_prim_sub_freq_daily",
  "shap_py_int_death_ed_attainment_primary", "shap_py_int_death_eva_fisica_any", "shap_py_int_death_eva_consumo_any",
  "shap_py_int_death_eva_sm_any", "shap_py_int_death_age_ns3_simpler", "shap_py_int_death_prim_sub_freq_high_simpler",
  "results_list", "death_ordinal_results", "death_ordinal_results2"
)
.pred22_files <- file.path(data_out, paste0(.pred22_latest_stem, "__", .pred22_suffixes, ".rds"))

.pred225_files <- file.path(data_out, c("pred22_ndp_2026_07_13__dualfits.rds", "pred225_metrics_2026_07_13.rds"))

.pred24_files <- file.path(data_out, c(
  "pred24_ndp_2026_07_14__dual_shap_ds_py.rds",
  "pred24_ndp_2026_07_14__dual_shap_ds_upd_py.rds",
  "pred24_ndp_2026_07_14__dual_shap_ds_int_py.rds",
  "pred24_ndp_2026_07_14__results_funcform_readm.rds",
  "pred24_ndp_2026_07_14__results_funcform_death.rds"
))

.all_inputs <- c(
  setNames(.pred225_files, rep("prediction225", length(.pred225_files))),
  setNames(.pred22_files, rep("prediction22", length(.pred22_files))),
  setNames(.pred24_files, rep("prediction24", length(.pred24_files)))
)

.missing <- .all_inputs[!file.exists(.all_inputs)]
if (length(.missing)) {
  stop("Missing required input file(s):\n", paste("  -", .missing, collapse = "\n"), call. = FALSE)
}
stopifnot(!anyDuplicated(.all_inputs))
stopifnot(!any(file.info(.all_inputs)$size == 0))

cat("Computing MD5 for", length(.all_inputs), "input files (this can take a few minutes for the larger ones)...\n")
.t_md5 <- Sys.time()
.md5 <- tools::md5sum(.all_inputs)
cat(sprintf("MD5 computed in %.1f min.\n", as.numeric(difftime(Sys.time(), .t_md5, units = "mins"))))

.fi <- file.info(.all_inputs)
input_manifest <- data.frame(
  section = names(.all_inputs),
  file = basename(.all_inputs),
  size_MiB = round(.fi$size / 2^20, 2),
  mtime = as.character(.fi$mtime),
  md5 = unname(.md5),
  stringsAsFactors = FALSE
)
rownames(input_manifest) <- NULL

write.csv(input_manifest, file.path(patch_out, "input_manifest_2026_07_22.csv"), row.names = FALSE)
print(input_manifest[, c("section", "file", "size_MiB")], row.names = FALSE)
cat(sprintf("\nChunk 04 PASS: %d input files inventoried, none duplicated, none zero-byte.\n", nrow(input_manifest)))

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-04-input-manifest] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-05-load-pred225
#| message: false

.t0 <- proc.time()

dualfits_path <- file.path(data_out, "pred22_ndp_2026_07_13__dualfits.rds")
metrics225_path <- file.path(data_out, "pred225_metrics_2026_07_13.rds")
stopifnot(file.exists(dualfits_path), file.exists(metrics225_path))

cat("Loading dualfits (~550 MiB, this is the largest single input)...\n")
bundle225 <- readRDS(dualfits_path)
stopifnot(all(c("dual_fits", "specs", "GRID") %in% names(bundle225)))
dual_fits <- bundle225$dual_fits
specs     <- bundle225$specs
GRID      <- bundle225$GRID

stopifnot(setequal(names(dual_fits), c("base", "updated", "updated2", "shap", "interact")))
stopifnot(isTRUE(bundle225$engine_fixed_strata))
stopifnot(identical(as.numeric(GRID), c(3, 6, 12, 36, 60)))
for (nm in names(dual_fits)) {
  stopifnot(length(dual_fits[[nm]]$raw_predictions) == 25L)
}
cat("PASS: 5 models (base, updated, updated2, shap, interact), 25 OOF blocks each,\n")
cat("      GRID = 3,6,12,36,60 months, engine_fixed_strata = TRUE.\n")

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-05-load-pred225] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-06-ibs-corrected-pred225
#| message: false

.t0 <- proc.time()

# recompute_dual_cox_ibs_from_raw() re-derives $metrics/$summary from the SAME
# stored raw_predictions; no Cox model is refit. This is the block-average
# summary (each of the 25 OOF blocks scored, then averaged) -- a companion
# but numerically distinct aggregation from the pooled-subject dIBS in the
# next chunk (small differences between the two are expected, see chunk 07).
ibs225_rows <- list()
dual_fits_corrected <- list()
for (nm in names(dual_fits)) {
  corrected <- recompute_dual_cox_ibs_from_raw(dual_fits[[nm]], outcomes = c("readmission", "death"), verbose = FALSE)
  stopifnot(identical(corrected$config$ipcw_order_contract, IPCW_ORDER_CONTRACT))
  dual_fits_corrected[[nm]] <- corrected
  s <- corrected$summary
  s <- s[s$Metric == "IBS", , drop = FALSE]
  s$Model <- nm
  ibs225_rows[[nm]] <- s[, c("Model", "Risk", "Time", "mean", "sd", "n")]
}
pred225_ibs_corrected_by_model_horizon <- do.call(rbind, ibs225_rows)
rownames(pred225_ibs_corrected_by_model_horizon) <- NULL

write.csv(pred225_ibs_corrected_by_model_horizon,
          file.path(patch_out, "pred225_ibs_corrected_by_model_horizon.csv"), row.names = FALSE)

cat("Corrected IBS, Global (block-averaged over the 25 OOF blocks):\n")
g <- pred225_ibs_corrected_by_model_horizon[pred225_ibs_corrected_by_model_horizon$Time == "Global", ]
print(g[order(g$Risk, g$Model), c("Model", "Risk", "mean")], row.names = FALSE)

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-06-ibs-corrected-pred225] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-07-paired-delta-ibs-pred225
#| message: false

.t0 <- proc.time()

# Helper functions copied verbatim from prediction225_converted_mod.ipynb,
# chunks 'deltaCIBS-cv-engine' and 'deltaCIBS-cv-run' (already corrected:
# both use match(requested_times, unique_times) to restore patient order).
.delta_ibs_G_ordered <- function(sf, tt, g_min = .05) {
  requested_times <- pmax(as.numeric(tt), 0); unique_times <- sort(unique(requested_times))
  g_unique <- as.numeric(summary(sf, times = unique_times, extend = TRUE)$surv)
  if (length(g_unique) != length(unique_times)) stop("Unexpected censoring-survival lookup length.")
  lookup_index <- match(requested_times, unique_times)
  if (anyNA(lookup_index)) stop("Could not restore patient-time order in IPCW lookup.")
  pmax(g_unique[lookup_index], g_min)
}
.ibs_prep <- function(tr_time, tr_event, te_time, eval_times, g_min = .05, eps = 1e-8) {
  ev <- as.numeric(tr_event); ev[is.na(ev)] <- 0; sf <- survival::survfit(survival::Surv(tr_time, 1 - ev) ~ 1)
  safe <- sf$time[sf$surv >= g_min]; tau <- if (length(safe)) max(safe) else max(eval_times)
  keep <- eval_times <= tau & eval_times <= max(tr_time, na.rm = TRUE); tu <- eval_times[keep]
  Gat <- function(tt) .delta_ibs_G_ordered(sf, tt, g_min)
  list(times_use = tu, keep = keep, gt0 = if (length(tu)) 1 / Gat(tu) else numeric(0), gT = 1 / Gat(pmax(te_time - eps, 0)))
}
ibs_fast <- function(S_keep, te_time, te_event, idx, prep) {
  tu <- prep$times_use; if (length(tu) < 2L) return(NA_real_)
  Tt <- te_time[idx]; Dt <- te_event[idx]; gT <- prep$gT[idx]; S <- S_keep[idx, , drop = FALSE]
  bs <- vapply(seq_along(tu), function(j) { t0 <- tu[j]; S0 <- S[, j]; ar <- Tt > t0; fl <- (!ar) & (Dt == 1)
    w <- numeric(length(Tt)); w[ar] <- prep$gt0[j] * (1 - S0[ar])^2; w[fl] <- gT[fl] * (S0[fl])^2; mean(w, na.rm = TRUE) }, numeric(1))
  den <- max(tu) - min(tu); if (!is.finite(den) || den <= 0) return(NA_real_)
  sum(diff(tu) * (head(bs, -1L) + tail(bs, -1L)) / 2) / den
}
.cv_extract <- function(res, outcome = c("readmission", "death")) {
  outcome <- match.arg(outcome)
  rp <- res$raw_predictions
  if (is.null(rp) || !length(rp)) stop("no raw_predictions (stripped or calibration object, not a dual eval)")
  cfg <- res$config; et <- as.numeric(cfg$eval_times)
  N <- max(vapply(rp, function(b) max(b$original_val_idx), 1L))
  imps <- sort(unique(vapply(rp, function(b) b$imp_idx, 1L)))
  time_v <- rep(NA_real_, N); event_v <- rep(NA_real_, N); lp_imp <- list(); S_imp <- list()
  for (im in imps) {
    lp <- rep(NA_real_, N); S <- matrix(NA_real_, N, length(et))
    for (b in rp) { if (b$imp_idx != im) next
      o <- b[[outcome]]; if (is.null(o) || !is.null(o$error) || is.null(o$lp_val)) next
      idx <- b$original_val_idx; lp[idx] <- as.numeric(o$lp_val); S[idx, ] <- as.matrix(o$surv_val_matrix)
      yv <- o$y_val; time_v[idx] <- as.numeric(yv[, 1]); event_v[idx] <- as.numeric(yv[, 2]) }
    lp_imp[[length(lp_imp) + 1L]] <- lp; S_imp[[length(S_imp) + 1L]] <- S }
  list(time = time_v, event = event_v, lp = Reduce(`+`, lp_imp) / length(lp_imp),
       S = Reduce(`+`, S_imp) / length(S_imp), eval_times = et)
}
deltaIBS_boot_x <- function(A, Z, B = 2000L, seed = 2125L) {
  ok <- is.finite(A$lp) & is.finite(Z$lp) & is.finite(A$time) & is.finite(A$event) &
    rowSums(is.finite(A$S)) == ncol(A$S) & rowSums(is.finite(Z$S)) == ncol(Z$S)
  ti <- A$time[ok]; ev <- A$event[ok]; SA <- A$S[ok, , drop = FALSE]; SZ <- Z$S[ok, , drop = FALSE]
  prep <- .ibs_prep(ti, ev, ti, A$eval_times)  # censoring curve from the OOF cohort itself, fixed within the bootstrap
  SAk <- SA[, prep$keep, drop = FALSE]; SZk <- SZ[, prep$keep, drop = FALSE]; n <- nrow(SAk); i0 <- seq_len(n)
  iA <- ibs_fast(SAk, ti, ev, i0, prep); iZ <- ibs_fast(SZk, ti, ev, i0, prep); set.seed(seed); d <- numeric(B)
  for (b in seq_len(B)) { idx <- sample.int(n, n, TRUE); d[b] <- ibs_fast(SAk, ti, ev, idx, prep) - ibs_fast(SZk, ti, ev, idx, prep) }
  q <- quantile(d, c(.025, .975), na.rm = TRUE, names = FALSE)
  data.frame(IBS_A = iA, IBS_B = iZ, dIBS = iA - iZ,
             dIBS_lo = q[1], dIBS_hi = q[2], excludes_0 = isTRUE(q[1] > 0 || q[2] < 0), n = n)
}

# rank_lvl / rk() / fml_txt() / build_pairs() copied verbatim from
# prediction225_converted_mod.ipynb, chunk 'deltaCIBS-cv-run'.
fml_txt <- function(nm, oc) { i <- if (oc == "readmission") 1L else 2L; paste(deparse(specs[[nm]][[i]]), collapse = " ") }
rank_lvl <- c(base = 1, updated = 2, updated2 = 3, shap = 4, interact = 5, int_corr = 6)
rk <- function(nm) { r <- rank_lvl[nm]; if (is.na(r)) 99L else as.integer(r) }
mods <- names(dual_fits)
build_pairs <- function(oc_label) {
  oc <- tolower(oc_label); out <- list(); skipped <- character(0)
  for (p in utils::combn(mods, 2, simplify = FALSE)) {
    p <- p[order(vapply(p, rk, 1L))]  # A = simpler/earlier model, B = more complex
    if (identical(fml_txt(p[1], oc), fml_txt(p[2], oc))) { skipped <- c(skipped, sprintf("%s~%s", p[1], p[2])); next }
    out[[length(out) + 1L]] <- c(p[1], p[2], oc_label)
  }
  if (length(skipped)) cat(sprintf("[%s] skipped identical formula: %s\n", oc_label, paste(skipped, collapse = ", ")))
  out
}
cmp <- c(build_pairs("Readmission"), build_pairs("Death"))
cat(sprintf("Running %d paired dIBS comparisons, B = 500 ...\n", length(cmp)))

RNGkind("Mersenne-Twister", "Inversion", "Rejection")
B_DELTA <- 500L
needed <- unique(do.call(rbind, lapply(cmp, function(z) data.frame(nm = c(z[1], z[2]), oc = tolower(z[3]), stringsAsFactors = FALSE))))
EX <- new.env()
for (r in seq_len(nrow(needed))) {
  k <- paste(needed$nm[r], needed$oc[r])
  EX[[k]] <- .cv_extract(dual_fits[[needed$nm[r]]], needed$oc[r])
}
getx <- function(nm, oc) EX[[paste(nm, oc)]]

t0 <- Sys.time()
delta_rows <- lapply(seq_along(cmp), function(i) {
  z <- cmp[[i]]; oc <- tolower(z[3]); A <- getx(z[1], oc); Z <- getx(z[2], oc)
  di <- deltaIBS_boot_x(A, Z, B = B_DELTA)
  cat(sprintf("  [%d/%d] %s: %s vs %s  (%.1fs elapsed)\n", i, length(cmp), z[3], z[1], z[2],
              as.numeric(difftime(Sys.time(), t0, units = "secs"))))
  data.frame(outcome = z[3], A = z[1], B = z[2],
             IBS_A = di$IBS_A, IBS_B = di$IBS_B, dIBS = di$dIBS,
             dIBS_lo = di$dIBS_lo, dIBS_hi = di$dIBS_hi,
             dIBS_favors = ifelse(di$excludes_0, ifelse(di$dIBS < 0, z[1], z[2]), "ns"),
             n = di$n, stringsAsFactors = FALSE)
})
cat(sprintf("Completed %d comparisons in %.1fs\n", length(cmp), as.numeric(difftime(Sys.time(), t0, units = "secs"))))
delta_ibs_tbl <- do.call(rbind, delta_rows)

# The old delta_tbl's dC/C_A/C_B columns are untouched by this bug (they come
# from the linear predictor, not the IPCW censoring lookup) and are copied
# across unchanged; only the IBS-related columns are replaced.
old_bundle225 <- readRDS(metrics225_path)
stopifnot("delta_tbl" %in% names(old_bundle225))
old_delta_tbl <- old_bundle225$delta_tbl
stopifnot(nrow(old_delta_tbl) == 20L, ncol(old_delta_tbl) == 14L)
keep_cols <- c("outcome", "A", "B", "C_A", "C_B", "dC", "dC_CI", "dC_favors", "n")
stopifnot(all(keep_cols %in% names(old_delta_tbl)))
pred225_delta_ibs_corrected <- merge(old_delta_tbl[, keep_cols], delta_ibs_tbl, by = c("outcome", "A", "B"), all = TRUE, suffixes = c("", "_new"))
stopifnot(nrow(pred225_delta_ibs_corrected) == 20L)
pred225_delta_ibs_corrected$dIBS_CI <- sprintf("[%.4f, %.4f]", pred225_delta_ibs_corrected$dIBS_lo, pred225_delta_ibs_corrected$dIBS_hi)
pred225_delta_ibs_corrected <- pred225_delta_ibs_corrected[, c(
  "outcome", "A", "B", "C_A", "C_B", "dC", "dC_CI", "dC_favors",
  "IBS_A", "IBS_B", "dIBS", "dIBS_CI", "dIBS_favors", "n"
)]
pred225_delta_ibs_corrected <- pred225_delta_ibs_corrected[order(pred225_delta_ibs_corrected$outcome, -abs(pred225_delta_ibs_corrected$dC)), ]
rownames(pred225_delta_ibs_corrected) <- NULL

write.csv(pred225_delta_ibs_corrected, file.path(patch_out, "pred225_delta_ibs_corrected_B500.csv"), row.names = FALSE)
saveRDS(pred225_delta_ibs_corrected, file.path(patch_out, "pred225_delta_ibs_corrected_B500.rds"))

cat("\n=== Corrected paired dC / dIBS table (20 rows, 14 cols) ===\n")
print(pred225_delta_ibs_corrected, row.names = FALSE)

cat("\nHeadline comparison for model selection, updated2 vs shap:\n")
print(pred225_delta_ibs_corrected[
  (pred225_delta_ibs_corrected$A == "updated2" & pred225_delta_ibs_corrected$B == "shap") |
  (pred225_delta_ibs_corrected$A == "shap" & pred225_delta_ibs_corrected$B == "updated2"), ],
  row.names = FALSE)

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-07-paired-delta-ibs-pred225] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-08-registry-pred22
#| message: false

.t0 <- proc.time()
options(dualcox.concordance_score = "risk", dualcox.concordance_within_strata = FALSE)

# The newest pred22 image is used only for its FILENAME (to build sidecar
# paths); the 5.6 GB .Rdata itself is never opened by this notebook.
.rdata_files <- list.files(data_out, pattern = "^pred22_ndp_.*\\.Rdata$", full.names = TRUE, ignore.case = TRUE)
stopifnot(length(.rdata_files) > 0)
.latest_rdata <- .rdata_files[order(file.info(.rdata_files)$mtime, decreasing = TRUE)][1]
.latest_stem  <- tools::file_path_sans_ext(basename(.latest_rdata))
cat("Newest pred22 image (name only, NOT loaded):", basename(.latest_rdata), "\n")

ff_cache <- new.env(parent = emptyenv())
ff_exists <- function(name) exists(name, envir = ff_cache, inherits = FALSE)
ff_get <- function(name) get(name, envir = ff_cache, inherits = FALSE)

# Copied verbatim from prediction22_converted_mod.ipynb, chunk
# 'functional-forms-risk-cindex-setup'.
ff_sidecar_map <- c(results_shap_xgb_py_upd_post_int_alt1 = "dual_shap_int_d_fullph", results_shap_xgb_py_upd_post_int_alt2 = "dual_shap_int_d_shap_rule", results_shap_py_int_readm_age_ns3 = "shap_py_int_readm_age_ns3", results_shap_py_int_readm_dit_ns3 = "shap_py_int_readm_dit_ns3", results_shap_py_int_readm_dit_quad = "shap_py_int_readm_dit_quad", results_shap_py_int_readm_dit_hinge5 = "shap_py_int_readm_dit_hinge5", results_shap_py_int_readm_dit_log = "shap_py_int_readm_dit_log", results_shap_py_int_readm_pobr_ns3 = "shap_py_int_readm_pobr_ns3", results_shap_py_int_readm_pobr_hinge015 = "shap_py_int_readm_pobr_hinge015", results_shap_py_int_readm_eva_consumo_any = "shap_py_int_readm_eva_consumo_any", results_shap_py_int_readm_eva_sm_any = "shap_py_int_readm_eva_sm_any", results_shap_py_int_death_age_ns3 = "shap_py_int_death_age_ns3", results_shap_py_int_death_age_quad = "shap_py_int_death_age_quad", results_shap_py_int_death_dit_hinge_9 = "shap_py_int_death_dit_hinge_9", results_shap_py_int_death_pobr_hinge022 = "shap_py_int_death_pobr_hinge022", results_shap_py_int_death_tenure_hinge1 = "shap_py_int_death_tenure_hinge1", results_shap_py_int_death_prim_sub_freq_daily = "shap_py_int_death_prim_sub_freq_daily", results_shap_py_int_death_ed_attainment_primary = "shap_py_int_death_ed_attainment_primary", results_shap_py_int_death_eva_fisica_any = "shap_py_int_death_eva_fisica_any", results_shap_py_int_death_eva_consumo_any = "shap_py_int_death_eva_consumo_any", results_shap_py_int_death_eva_sm_any = "shap_py_int_death_eva_sm_any", results_shap_py_int_death_age_ns3_simpler = "shap_py_int_death_age_ns3_simpler", results_shap_py_int_death_prim_sub_freq_high_simpler = "shap_py_int_death_prim_sub_freq_high_simpler", results_list = "results_list", death_ordinal_results = "death_ordinal_results", death_ordinal_results2 = "death_ordinal_results2")

for (nm in names(ff_sidecar_map)) {
  path <- file.path(data_out, paste0(.latest_stem, "__", ff_sidecar_map[[nm]], ".rds"))
  if (!file.exists(path)) {
    stop("Required sidecar missing (this notebook will NOT fall back to the 5.6 GB .Rdata): ", path, call. = FALSE)
  }
  cat("[RDS] Loading", basename(path), "as", nm, "\n")
  assign(nm, readRDS(path), envir = ff_cache)
}
cat("All", length(ff_sidecar_map), "pred22 sidecars loaded without touching the full .Rdata.\n")

# Registry copied verbatim from prediction22_converted_mod.ipynb, chunk
# 'functional-forms-risk-cindex-engine2'.
ff_registry <- data.frame(id = c("readm_linear", "readm_age_ns3", "readm_dit_ns3", "readm_dit_quad", "readm_dit_hinge5", "readm_dit_log", "readm_pobr_ns3", "readm_pobr_hinge015", "readm_eva_consumo_any", "readm_eva_sm_any", "death_full_linear", "death_full_age_ns3", "death_full_age_quad", "death_full_dit_hinge9", "death_full_pobr_hinge022", "death_full_tenure_hinge1", "death_full_prim_freq_daily", "death_full_ed_primary", "death_full_eva_fisica_any", "death_full_eva_consumo_any", "death_full_eva_sm_any", "death_rule2_linear", "death_rule2_age_ns3", "death_rule2_prim_freq_high"), label = c("Linear baseline", "Age natural spline, df=3", "Treatment duration natural spline", "Treatment duration quadratic", "Treatment duration hinge at 5 months", "Treatment duration log1p", "Poverty natural spline", "Poverty hinge at 0.15", "Substance-use discharge evaluation, any lower achievement", "Mental-health discharge evaluation, any lower achievement", "Linear baseline", "Age natural spline, df=3", "Age quadratic", "Treatment duration hinge at 9 months", "Poverty hinge at 0.22", "Household tenure hinge at 1", "Daily primary-substance-use indicator", "Primary school or less indicator", "Physical-health discharge evaluation, any lower achievement", "Substance-use discharge evaluation, any lower achievement", "Mental-health discharge evaluation, any lower achievement", "Rule-2 baseline", "Age natural spline, df=3", "High primary-substance-use frequency indicator"), object_name = c("results_shap_xgb_py_upd_post_int_alt1", "results_shap_py_int_readm_age_ns3", "results_shap_py_int_readm_dit_ns3", "results_shap_py_int_readm_dit_quad", "results_shap_py_int_readm_dit_hinge5", "results_shap_py_int_readm_dit_log", "results_shap_py_int_readm_pobr_ns3", "results_shap_py_int_readm_pobr_hinge015", "results_shap_py_int_readm_eva_consumo_any", "results_shap_py_int_readm_eva_sm_any", "results_shap_xgb_py_upd_post_int_alt1", "results_shap_py_int_death_age_ns3", "results_shap_py_int_death_age_quad", "results_shap_py_int_death_dit_hinge_9", "results_shap_py_int_death_pobr_hinge022", "results_shap_py_int_death_tenure_hinge1", "results_shap_py_int_death_prim_sub_freq_daily", "results_shap_py_int_death_ed_attainment_primary", "results_shap_py_int_death_eva_fisica_any", "results_shap_py_int_death_eva_consumo_any", "results_shap_py_int_death_eva_sm_any", "results_shap_xgb_py_upd_post_int_alt2", "results_shap_py_int_death_age_ns3_simpler", "results_shap_py_int_death_prim_sub_freq_high_simpler"), outcome = c(rep("Readmission", 10), rep("Death", 14)), outcome_key = c(rep("readmission", 10), rep("death", 14)), family = c(rep("Readmission", 10), rep("Death full PH", 11), rep("Death rule 2", 3)), baseline_id = c(rep("readm_linear", 10), rep("death_full_linear", 11), rep("death_rule2_linear", 3)), is_baseline = c(TRUE, rep(FALSE, 9), TRUE, rep(FALSE, 10), TRUE, rep(FALSE, 2)), stringsAsFactors = FALSE)

ff_register_nested <- function(container_name, slot = NULL, outcome, outcome_key, family, baseline_id, prefix) { if (!ff_exists(container_name)) return(NULL); ff_container <- ff_get(container_name); ff_list <- if (is.null(slot)) ff_container else ff_container[[slot]]; if (is.null(ff_list) || !length(ff_list)) return(NULL); ff_names <- names(ff_list); if (is.null(ff_names)) ff_names <- paste0("variant_", seq_along(ff_list)); do.call(rbind, lapply(seq_along(ff_list), function(i) { ff_safe <- gsub("[^A-Za-z0-9]+", "_", ff_names[i]); ff_object_name <- paste0("ff_nested_", prefix, "_", ff_safe); assign(ff_object_name, ff_list[[i]], envir = ff_cache); data.frame(id = paste0(prefix, "_", ff_safe), label = paste0("Ordinal score: ", gsub("_", " ", ff_names[i])), object_name = ff_object_name, outcome = outcome, outcome_key = outcome_key, family = family, baseline_id = baseline_id, is_baseline = FALSE, stringsAsFactors = FALSE) })) }
ff_nested_rows <- Filter(Negate(is.null), list(ff_register_nested("results_list", outcome = "Readmission", outcome_key = "readmission", family = "Readmission", baseline_id = "readm_linear", prefix = "readm_ordinal"), ff_register_nested("death_ordinal_results", slot = "results", outcome = "Death", outcome_key = "death", family = "Death full PH", baseline_id = "death_full_linear", prefix = "death_full_ordinal"), ff_register_nested("death_ordinal_results2", slot = "results", outcome = "Death", outcome_key = "death", family = "Death rule 2", baseline_id = "death_rule2_linear", prefix = "death_rule2_ordinal")))
if (length(ff_nested_rows)) ff_registry <- rbind(ff_registry, do.call(rbind, ff_nested_rows))
ff_has_raw <- function(object_name) { if (!ff_exists(object_name)) return(FALSE); ff_object <- ff_get(object_name); is.list(ff_object) && is.list(ff_object$raw_predictions) && length(ff_object$raw_predictions) > 0L }
ff_registry$available <- vapply(ff_registry$object_name, ff_has_raw, logical(1))
if (any(!ff_registry$available)) cat("Unavailable or stripped result objects omitted:", paste(ff_registry$object_name[!ff_registry$available], collapse = ", "), "\n")
ff_registry <- ff_registry[ff_registry$available, setdiff(names(ff_registry), "available"), drop = FALSE]
stopifnot(!anyDuplicated(ff_registry$id))
stopifnot(all(c("readm_linear", "death_full_linear", "death_rule2_linear") %in% ff_registry$id))
stopifnot(nrow(ff_registry) == 41L)

cat(sprintf("PASS: registry has %d entries (%d baselines, %d functional-form variants).\n",
            nrow(ff_registry), sum(ff_registry$is_baseline), sum(!ff_registry$is_baseline)))
print(ff_registry[, c("id", "label", "family", "baseline_id")], row.names = FALSE)

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-08-registry-pred22] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-09-ibs-rescoring-pred22
#| message: false

.t0 <- proc.time()

FF_HORIZONS <- c(6, 12, 36, 60)
.by_fold_rows <- list(); .by_horizon_rows <- list()
.recompute_cache <- new.env(parent = emptyenv())  # keyed by object_name; the two baselines each serve two registry rows
get_corrected <- function(object_name) {
  if (!exists(object_name, envir = .recompute_cache, inherits = FALSE)) {
    obj <- ff_get(object_name)
    corrected <- recompute_dual_cox_ibs_from_raw(obj, outcomes = c("readmission", "death"), verbose = FALSE)
    assign(object_name, corrected, envir = .recompute_cache)
  }
  get(object_name, envir = .recompute_cache, inherits = FALSE)
}

for (i in seq_len(nrow(ff_registry))) {
  row <- ff_registry[i, ]
  corrected <- get_corrected(row$object_name)

  m <- corrected$metrics
  m <- m[m$Risk == row$outcome & m$Metric == "IBS" & m$Time %in% as.character(FF_HORIZONS), , drop = FALSE]
  m$id <- row$id; m$family <- row$family; m$is_baseline <- row$is_baseline
  .by_fold_rows[[row$id]] <- m[, c("id", "family", "is_baseline", "Imp", "Fold", "Risk", "Time", "Value")]

  s <- corrected$summary
  s <- s[s$Risk == row$outcome & s$Metric == "IBS" & s$Time %in% as.character(FF_HORIZONS), , drop = FALSE]
  s$id <- row$id; s$family <- row$family; s$is_baseline <- row$is_baseline
  .by_horizon_rows[[row$id]] <- s[, c("id", "family", "is_baseline", "Risk", "Time", "mean", "sd", "q025", "q975", "n")]
}
pred22_ibs_corrected_by_fold <- do.call(rbind, .by_fold_rows); rownames(pred22_ibs_corrected_by_fold) <- NULL
pred22_ibs_corrected_by_model_horizon <- do.call(rbind, .by_horizon_rows); rownames(pred22_ibs_corrected_by_model_horizon) <- NULL

stopifnot(nrow(pred22_ibs_corrected_by_fold) == 4100L)          # 41 models x 25 OOF blocks x 4 horizons
stopifnot(nrow(pred22_ibs_corrected_by_model_horizon) == 164L)  # 41 models x 4 horizons
stopifnot(!any(abs(pred22_ibs_corrected_by_model_horizon$mean[pred22_ibs_corrected_by_model_horizon$Risk == "Readmission"] - 0.32) < 0.02))
stopifnot(!any(abs(pred22_ibs_corrected_by_model_horizon$mean[pred22_ibs_corrected_by_model_horizon$Risk == "Death"] - 0.067) < 0.01))

write.csv(pred22_ibs_corrected_by_fold, file.path(patch_out, "pred22_ibs_corrected_by_fold.csv"), row.names = FALSE)
write.csv(pred22_ibs_corrected_by_model_horizon, file.path(patch_out, "pred22_ibs_corrected_by_model_horizon.csv"), row.names = FALSE)

cat(sprintf("PASS: %d fold rows, %d model-horizon rows; no stale legacy-scale IBS values found.\n",
            nrow(pred22_ibs_corrected_by_fold), nrow(pred22_ibs_corrected_by_model_horizon)))

cat("\nBaseline footprint (corrected IBS at 6/12/36/60 months):\n")
for (bl in c("readm_linear", "death_full_linear", "death_rule2_linear")) {
  cat(" ", bl, ":\n")
  print(pred22_ibs_corrected_by_model_horizon[pred22_ibs_corrected_by_model_horizon$id == bl, c("Risk", "Time", "mean")], row.names = FALSE)
}

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-09-ibs-rescoring-pred22] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-10-functional-forms-pred22
#| message: false

.t0 <- proc.time()

# Copied verbatim from prediction22_converted_mod.ipynb, chunks
# 'functional-forms-risk-cindex-engine2' and 'functional-forms-risk-cindex-run'.
ff_match_horizon <- function(horizon, eval_times, tolerance = 1e-8) { ff_index <- which(abs(as.numeric(eval_times) - horizon) < tolerance); if (length(ff_index) != 1L) stop("Horizon ", horizon, " was not found exactly once in eval_times: ", paste(eval_times, collapse = ", "), call. = FALSE); ff_index }
ff_safe_uno_risk <- function(time, event, risk, horizon) { ff_ok <- is.finite(time) & is.finite(event) & is.finite(risk); ff_data <- data.frame(time = as.numeric(time[ff_ok]), event = as.numeric(event[ff_ok]), risk = as.numeric(risk[ff_ok])); if (nrow(ff_data) < 2L || sum(ff_data$event == 1 & ff_data$time <= horizon) < 2L) return(NA_real_); tryCatch(as.numeric(survival::concordance(survival::Surv(time, event) ~ risk, data = ff_data, timewt = "n/G2", reverse = TRUE, ymax = horizon)$concordance), error = function(e) { warning("C-index failed at ", horizon, " months: ", conditionMessage(e), call. = FALSE); NA_real_ }) }
ff_score_result_folds <- function(result, outcome_key, horizons) { ff_raw <- result$raw_predictions; if (is.null(ff_raw) || !length(ff_raw)) stop("raw_predictions are unavailable.", call. = FALSE); do.call(rbind, lapply(seq_along(ff_raw), function(i) { ff_block <- ff_raw[[i]]; ff_outcome <- ff_block[[outcome_key]]; if (is.null(ff_outcome) || !is.null(ff_outcome$error) || is.null(ff_outcome$surv_val_matrix)) return(NULL); ff_times <- as.numeric(ff_block$eval_times); ff_survival <- as.matrix(ff_outcome$surv_val_matrix); ff_y <- ff_outcome$y_val; ff_train_max <- max(as.numeric(ff_outcome$y_train[, 1]), na.rm = TRUE); do.call(rbind, lapply(horizons, function(ff_horizon) { ff_j <- ff_match_horizon(ff_horizon, ff_times); ff_c <- if (is.finite(ff_train_max) && ff_horizon < ff_train_max) ff_safe_uno_risk(as.numeric(ff_y[, 1]), as.numeric(ff_y[, 2]), pmin(pmax(1 - ff_survival[, ff_j], 0), 1), ff_horizon) else NA_real_; data.frame(Imp = as.integer(ff_block$imp_idx), Fold = as.integer(ff_block$fold_idx), horizon = ff_horizon, C_index = ff_c, n_validation = nrow(ff_survival), events_by_horizon = sum(as.numeric(ff_y[, 2]) == 1 & as.numeric(ff_y[, 1]) <= ff_horizon, na.rm = TRUE), stringsAsFactors = FALSE) })) })) }
ff_fold_score_cache <- new.env(parent = emptyenv())
ff_get_fold_scores <- function(model_id, horizons) { ff_cache_key <- paste0(model_id, "__", paste(horizons, collapse = "_")); if (!exists(ff_cache_key, envir = ff_fold_score_cache, inherits = FALSE)) { ff_row <- ff_registry[match(model_id, ff_registry$id), , drop = FALSE]; if (!nrow(ff_row)) stop("Unknown model ID: ", model_id, call. = FALSE); cat("[SCORE]", model_id, "from", ff_row$object_name, "\n"); assign(ff_cache_key, ff_score_result_folds(ff_get(ff_row$object_name), ff_row$outcome_key, horizons), envir = ff_fold_score_cache) }; get(ff_cache_key, envir = ff_fold_score_cache, inherits = FALSE) }
ff_ibs_summary_cache <- new.env(parent = emptyenv())
ff_extract_ibs_summary <- function(result, outcome, horizons, cache_key = NULL) {
  if (is.null(cache_key)) cache_key <- paste(c(result$config$formula_readmit, result$config$formula_death, paste(result$config$eval_times, collapse = ","), result$config$timestamp), collapse = "|")
  key <- paste(cache_key, outcome, sep = "::")
  if (!exists(key, envir = ff_ibs_summary_cache, inherits = FALSE)) {
    outcome_key <- if (outcome == "Readmission") "readmission" else "death"
    corrected <- recompute_dual_cox_ibs_from_raw(result, outcomes = outcome_key, verbose = FALSE)
    assign(key, corrected$summary, envir = ff_ibs_summary_cache)
  }
  ff_summary <- get(key, envir = ff_ibs_summary_cache, inherits = FALSE)
  ff_time <- suppressWarnings(as.numeric(as.character(ff_summary$Time)))
  ff_keep <- ff_summary$Risk == outcome & ff_summary$Metric == "IBS" & is.finite(ff_time) & ff_time %in% horizons
  data.frame(horizon = ff_time[ff_keep], IBS = as.numeric(ff_summary$mean[ff_keep]), stringsAsFactors = FALSE)
}

FF_C_GAIN_THRESHOLD <- 0.01
FF_IBS_WORSENING_TOLERANCE <- 0.001
ff_jobs <- ff_registry[!ff_registry$is_baseline, , drop = FALSE]
.ff_start_time <- Sys.time()
.ff_result_rows <- vector("list", nrow(ff_jobs))
for (i in seq_len(nrow(ff_jobs))) {
  ff_job <- ff_jobs[i, , drop = FALSE]
  cat(sprintf("[%d/%d] %s | %s vs %s\n", i, nrow(ff_jobs), ff_job$family, ff_job$id, ff_job$baseline_id))
  ff_variant_scores <- ff_get_fold_scores(ff_job$id, FF_HORIZONS)
  ff_baseline_scores <- ff_get_fold_scores(ff_job$baseline_id, FF_HORIZONS)
  ff_paired <- merge(ff_variant_scores, ff_baseline_scores, by = c("Imp", "Fold", "horizon"), suffixes = c("_variant", "_baseline"), all = FALSE, sort = TRUE)
  ff_paired$delta_C <- ff_paired$C_index_variant - ff_paired$C_index_baseline
  ff_c_summary <- do.call(rbind, lapply(split(ff_paired, ff_paired$horizon), function(x) data.frame(horizon = unique(x$horizon), C_variant = mean(x$C_index_variant, na.rm = TRUE), C_baseline = mean(x$C_index_baseline, na.rm = TRUE), delta_C = mean(x$delta_C, na.rm = TRUE), delta_C_sd = stats::sd(x$delta_C, na.rm = TRUE), delta_C_q025 = as.numeric(stats::quantile(x$delta_C, 0.025, na.rm = TRUE, names = FALSE)), delta_C_q975 = as.numeric(stats::quantile(x$delta_C, 0.975, na.rm = TRUE, names = FALSE)), n_pairs = sum(is.finite(x$delta_C)), mean_validation_n = mean(x$n_validation_variant, na.rm = TRUE), mean_events_by_horizon = mean(x$events_by_horizon_variant, na.rm = TRUE), stringsAsFactors = FALSE)))
  ff_variant_ibs <- ff_extract_ibs_summary(ff_get(ff_job$object_name), ff_job$outcome, FF_HORIZONS)
  ff_baseline_row <- ff_registry[match(ff_job$baseline_id, ff_registry$id), , drop = FALSE]
  ff_baseline_ibs <- ff_extract_ibs_summary(ff_get(ff_baseline_row$object_name), ff_baseline_row$outcome, FF_HORIZONS)
  names(ff_variant_ibs)[2] <- "IBS_variant"; names(ff_baseline_ibs)[2] <- "IBS_baseline"
  ff_metrics <- merge(ff_c_summary, merge(ff_variant_ibs, ff_baseline_ibs, by = "horizon", all = TRUE), by = "horizon", all.x = TRUE, sort = TRUE)
  ff_metrics$delta_IBS <- ff_metrics$IBS_variant - ff_metrics$IBS_baseline
  ff_metrics$id <- ff_job$id; ff_metrics$model <- ff_job$label; ff_metrics$family <- ff_job$family; ff_metrics$outcome <- ff_job$outcome; ff_metrics$baseline_id <- ff_job$baseline_id
  .ff_result_rows[[i]] <- ff_metrics
  cat(sprintf("    max mean delta C = %.5f | elapsed %.1f s\n", max(ff_metrics$delta_C, na.rm = TRUE), as.numeric(difftime(Sys.time(), .ff_start_time, units = "secs"))))
}
pred22_functional_forms_corrected <- do.call(rbind, .ff_result_rows)
pred22_functional_forms_corrected$C_gain_ge_0_01 <- pred22_functional_forms_corrected$delta_C >= FF_C_GAIN_THRESHOLD
pred22_functional_forms_corrected$IBS_not_materially_worse <- pred22_functional_forms_corrected$delta_IBS <= FF_IBS_WORSENING_TOLERANCE
pred22_functional_forms_corrected$meets_joint_rule <- pred22_functional_forms_corrected$C_gain_ge_0_01 & pred22_functional_forms_corrected$IBS_not_materially_worse
stopifnot(nrow(pred22_functional_forms_corrected) == 152L)
pred22_functional_forms_corrected <- pred22_functional_forms_corrected[order(pred22_functional_forms_corrected$family, pred22_functional_forms_corrected$horizon, -pred22_functional_forms_corrected$delta_C), , drop = FALSE]
rownames(pred22_functional_forms_corrected) <- NULL
pred22_functional_forms_joint_candidates <- pred22_functional_forms_corrected[pred22_functional_forms_corrected$meets_joint_rule, , drop = FALSE]

write.csv(pred22_functional_forms_corrected, file.path(patch_out, "pred22_functional_forms_corrected_152_comparisons.csv"), row.names = FALSE)
write.csv(pred22_functional_forms_joint_candidates, file.path(patch_out, "pred22_functional_forms_joint_candidates.csv"), row.names = FALSE)

cat(sprintf("\nResult: %d comparisons, %d joint-rule candidates (expected 152 and 0).\n",
            nrow(pred22_functional_forms_corrected), nrow(pred22_functional_forms_joint_candidates)))
if (nrow(pred22_functional_forms_joint_candidates)) {
  cat("Unexpected candidate(s) found -- inspect before relying on 'no functional-form change' as the conclusion:\n")
  print(pred22_functional_forms_joint_candidates, row.names = FALSE)
} else {
  cat("No functional-form variant increased absolute-risk Uno's C by >= 0.01 without materially worsening IBS.\n")
  cat("Functional-form selection from prediction22 remains unchanged by this patch.\n")
}

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-10-functional-forms-pred22] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-11-verify-pred24
#| message: false

.t0 <- proc.time()

# This chunk is read-only: it confirms that the five prediction24 sidecars
# already reproduce the ordered IPCW lookup by diffing stored vs recomputed
# IBS from stored OOF predictions. It writes a new RDS ONLY if a sidecar
# turns out to be 'repairable' (not expected here).
.diff_before_after <- function(result, label) {
  before <- result$summary
  before <- before[before$Metric == "IBS", c("Risk", "Time", "mean")]
  names(before)[3] <- "mean_before"
  corrected <- tryCatch(recompute_dual_cox_ibs_from_raw(result, outcomes = c("readmission", "death"), verbose = FALSE),
                         error = function(e) { cat("  [ERROR] recompute failed for", label, ":", conditionMessage(e), "\n"); NULL })
  if (is.null(corrected)) return(list(status = "unrepairable", detail = NULL))
  after <- corrected$summary
  after <- after[after$Metric == "IBS", c("Risk", "Time", "mean")]
  names(after)[3] <- "mean_after"
  merged <- merge(before, after, by = c("Risk", "Time"), all = TRUE)
  merged$abs_diff <- abs(merged$mean_after - merged$mean_before)
  merged$label <- label
  n_folds <- length(result$raw_predictions)
  max_diff <- max(merged$abs_diff, na.rm = TRUE)
  status <- if (is.finite(max_diff) && max_diff <= 1e-12) "already_corrected" else "repairable"
  cat(sprintf("  [%s] n_oof_blocks=%d max_abs_diff=%.3e -> %s\n", label, n_folds, max_diff, status))
  list(status = status, detail = merged, n_oof_blocks = n_folds, max_abs_diff = max_diff, corrected = corrected)
}

.verification_rows <- list(); .detail_rows <- list(); .repairable_objects <- list()
.record <- function(label, res) {
  .verification_rows[[label]] <<- data.frame(label = label, status = res$status,
                                               n_oof_blocks = if (is.null(res$n_oof_blocks)) NA_integer_ else res$n_oof_blocks,
                                               max_abs_diff = if (is.null(res$max_abs_diff)) NA_real_ else res$max_abs_diff,
                                               stringsAsFactors = FALSE)
  if (!is.null(res$detail)) .detail_rows[[label]] <<- res$detail
  if (identical(res$status, "repairable")) .repairable_objects[[label]] <<- res$corrected
}

# Main wrappers: results$lp, results$risk, comparison_summary
.main_files <- c(
  dual_shap_ds_py     = "pred24_ndp_2026_07_14__dual_shap_ds_py.rds",
  dual_shap_ds_upd_py = "pred24_ndp_2026_07_14__dual_shap_ds_upd_py.rds",
  dual_shap_ds_int_py = "pred24_ndp_2026_07_14__dual_shap_ds_int_py.rds"
)
for (nm in names(.main_files)) {
  path <- file.path(data_out, .main_files[[nm]])
  stopifnot(file.exists(path))
  cat("\nLoading", .main_files[[nm]], "...\n")
  wrapper <- readRDS(path)
  stopifnot(all(c("results", "comparison_summary") %in% names(wrapper)))
  stopifnot(all(c("lp", "risk") %in% names(wrapper$results)))
  for (branch in c("lp", "risk")) {
    .record(paste0(nm, "__", branch), .diff_before_after(wrapper$results[[branch]], paste0(nm, "__", branch)))
  }
  rm(wrapper); invisible(gc(FALSE))
}

# Functional-form containers: iterate by variant down to $result$results$lp / $risk.
# The variant path is discovered defensively rather than assumed, since the
# spec's own scope note only describes the expected shape, not a fixed count.
.ff_files <- c(
  results_funcform_readm = "pred24_ndp_2026_07_14__results_funcform_readm.rds",
  results_funcform_death = "pred24_ndp_2026_07_14__results_funcform_death.rds"
)
for (container_nm in names(.ff_files)) {
  path <- file.path(data_out, .ff_files[[container_nm]])
  stopifnot(file.exists(path))
  cat("\nLoading", .ff_files[[container_nm]], "...\n")
  container <- readRDS(path)
  variant_list <- NULL
  if (is.list(container) && !is.null(names(container)) &&
      all(vapply(container, function(v) is.list(v) && !is.null(v$result$results), logical(1)))) {
    variant_list <- container
  } else if (!is.null(container$results) && is.list(container$results) &&
             all(vapply(container$results, function(v) is.list(v) && !is.null(v$result$results), logical(1)))) {
    variant_list <- container$results
  }
  if (is.null(variant_list)) {
    cat("  [STRUCTURE NOT RECOGNIZED] top names:", paste(names(container), collapse = ", "), "-- recorded as unrepairable, no guessing.\n")
    .record(container_nm, list(status = "unrepairable", detail = NULL))
    rm(container); invisible(gc(FALSE)); next
  }
  cat("  n_variants =", length(variant_list), ":", paste(names(variant_list), collapse = ", "), "\n")
  for (variant_nm in names(variant_list)) {
    rr <- variant_list[[variant_nm]]$result$results
    for (branch in c("lp", "risk")) {
      res <- rr[[branch]]
      if (is.null(res)) { cat("  [MISSING BRANCH]", container_nm, variant_nm, branch, "\n"); next }
      .record(paste(container_nm, variant_nm, branch, sep = "__"), .diff_before_after(res, paste(container_nm, variant_nm, branch, sep = "__")))
    }
  }
  rm(container, variant_list); invisible(gc(FALSE))
}

pred24_ibs_order_verification <- do.call(rbind, .verification_rows); rownames(pred24_ibs_order_verification) <- NULL
pred24_ibs_before_after_full_precision <- if (length(.detail_rows)) { x <- do.call(rbind, .detail_rows); rownames(x) <- NULL; x } else data.frame()

write.csv(pred24_ibs_order_verification, file.path(patch_out, "pred24_ibs_order_verification.csv"), row.names = FALSE)
write.csv(pred24_ibs_before_after_full_precision, file.path(patch_out, "pred24_ibs_before_after_full_precision.csv"), row.names = FALSE)

cat("\n=== Verification summary ===\n")
print(table(pred24_ibs_order_verification$status))
cat(sprintf("Max abs diff across all checked branches: %.3e (already_corrected requires <= 1e-12)\n",
            max(pred24_ibs_order_verification$max_abs_diff, na.rm = TRUE)))

if (length(.repairable_objects)) {
  cat("\nUnexpected 'repairable' object(s) found -- saving a VERSIONED copy (original left untouched):\n")
  for (nm in names(.repairable_objects)) {
    out_path <- file.path(patch_out, paste0("pred24_", gsub("[^A-Za-z0-9]+", "_", nm), "_ibs_repaired_2026_07_22.rds"))
    saveRDS(.repairable_objects[[nm]], out_path)
    cat("  wrote", out_path, "\n")
  }
} else {
  cat("\nNo prediction24 sidecar required repair or a new RDS write, as expected.\n")
}
stopifnot(all(pred24_ibs_order_verification$status == "already_corrected"))

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-11-verify-pred24] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-12-bundle
#| message: false

.t0 <- proc.time()

run_params <- list(
  run_date = "2026-07-22",
  ipcw_order_contract = IPCW_ORDER_CONTRACT,
  refit_models = REFIT_MODELS,
  eval_times_grid = as.numeric(GRID),
  ff_horizons = FF_HORIZONS,
  b_delta = B_DELTA,
  seed = 2125L,
  estimand_note = paste(
    "IBS Global is the mean restricted error between the first and last node of",
    "eval_times (here, 3 to 60 months, divided by 57), NOT an IBS from 0 to 60",
    "months. All comparisons in this bundle use that same restricted grid."
  )
)

patch_bundle <- list(
  run_params = run_params,
  input_manifest = input_manifest,
  pred225_ibs_corrected_by_model_horizon = pred225_ibs_corrected_by_model_horizon,
  pred225_delta_ibs_corrected = pred225_delta_ibs_corrected,
  pred22_ibs_corrected_by_model_horizon = pred22_ibs_corrected_by_model_horizon,
  pred22_functional_forms_corrected = pred22_functional_forms_corrected,
  pred22_functional_forms_joint_candidates = pred22_functional_forms_joint_candidates,
  pred24_ibs_order_verification = pred24_ibs_order_verification,
  session_info = utils::capture.output(print(sessionInfo()))
)

.bundle_size_mib <- function(x) as.numeric(utils::object.size(x)) / 2^20
cat(sprintf("In-memory bundle size: %.2f MiB (must stay under 10 MiB; raw_predictions are never stored here)\n",
            .bundle_size_mib(patch_bundle)))
stopifnot(.bundle_size_mib(patch_bundle) < 10)

.tmp_rds <- file.path(patch_out, "ibs_patch_bundle_2026_07_22.rds.tmp")
.final_rds <- file.path(patch_out, "ibs_patch_bundle_2026_07_22.rds")
saveRDS(patch_bundle, .tmp_rds)

.reloaded <- readRDS(.tmp_rds)
stopifnot(identical(names(.reloaded), names(patch_bundle)))
stopifnot(nrow(.reloaded$pred225_delta_ibs_corrected) == 20L)
stopifnot(nrow(.reloaded$pred22_functional_forms_corrected) == 152L)
stopifnot(file.size(.tmp_rds) < 10 * 2^20)

if (file.exists(.final_rds) && !OVERWRITE) {
  stop("Final bundle already exists and OVERWRITE is FALSE: ", .final_rds, call. = FALSE)
}
file.rename(.tmp_rds, .final_rds)
cat("Bundle written and verified:", .final_rds, "\n")

manifest_final <- data.frame(
  file = basename(c(
    .final_rds,
    file.path(patch_out, "input_manifest_2026_07_22.csv"),
    file.path(patch_out, "pred225_ibs_corrected_by_model_horizon.csv"),
    file.path(patch_out, "pred225_delta_ibs_corrected_B500.csv"),
    file.path(patch_out, "pred22_ibs_corrected_by_fold.csv"),
    file.path(patch_out, "pred22_ibs_corrected_by_model_horizon.csv"),
    file.path(patch_out, "pred22_functional_forms_corrected_152_comparisons.csv"),
    file.path(patch_out, "pred22_functional_forms_joint_candidates.csv"),
    file.path(patch_out, "pred24_ibs_order_verification.csv"),
    file.path(patch_out, "pred24_ibs_before_after_full_precision.csv")
  )),
  stringsAsFactors = FALSE
)
manifest_final$path <- file.path(patch_out, manifest_final$file)
manifest_final$exists <- file.exists(manifest_final$path)
manifest_final$size_MiB <- ifelse(manifest_final$exists, round(file.info(manifest_final$path)$size / 2^20, 3), NA_real_)
manifest_final$md5 <- ifelse(manifest_final$exists, tools::md5sum(manifest_final$path), NA_character_)
write.csv(manifest_final[, c("file", "size_MiB", "md5")],
          file.path(patch_out, "ibs_patch_manifest_2026_07_22.csv"), row.names = FALSE)
print(manifest_final[, c("file", "exists", "size_MiB")], row.names = FALSE)
stopifnot(all(manifest_final$exists))

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-12-bundle] %.3f min\n", .elapsed))

In [ ]:
#| label: patch-13-checks
#| message: false

.t0 <- proc.time()

.check <- function(name, condition, detail = "") {
  data.frame(check = name, status = if (isTRUE(condition)) "PASS" else "FAIL", detail = detail, stringsAsFactors = FALSE)
}

.pred225_delta <- read.csv(file.path(patch_out, "pred225_delta_ibs_corrected_B500.csv"), stringsAsFactors = FALSE)
.pred22_fold <- read.csv(file.path(patch_out, "pred22_ibs_corrected_by_fold.csv"), stringsAsFactors = FALSE)
.pred22_horizon <- read.csv(file.path(patch_out, "pred22_ibs_corrected_by_model_horizon.csv"), stringsAsFactors = FALSE)
.pred22_ff <- read.csv(file.path(patch_out, "pred22_functional_forms_corrected_152_comparisons.csv"), stringsAsFactors = FALSE)
.pred22_ff_joint <- read.csv(file.path(patch_out, "pred22_functional_forms_joint_candidates.csv"), stringsAsFactors = FALSE)
.pred24_verif <- read.csv(file.path(patch_out, "pred24_ibs_order_verification.csv"), stringsAsFactors = FALSE)

# Per-model pooled Global IBS, reconstructed from the pairwise A/B table: each
# model's pooled IBS is recorded consistently across every pair it appears in
# (as IBS_A when it is the "A" side, IBS_B when it is the "B" side), so a
# single unique value per (outcome, model) should fall out here.
.pooled_ibs <- unique(rbind(
  setNames(.pred225_delta[, c("outcome", "A", "IBS_A")], c("outcome", "model", "ibs")),
  setNames(.pred225_delta[, c("outcome", "B", "IBS_B")], c("outcome", "model", "ibs"))
))
stopifnot(nrow(.pooled_ibs) == 10L)  # 5 models x 2 outcomes, each with one consistent pooled value

# Original July inputs must be byte-identical to what chunk 04 recorded.
.manifest_check_files <- c("pred22_ndp_2026_07_13__dualfits.rds", "pred225_metrics_2026_07_13.rds")
.manifest_path <- file.path(patch_out, "input_manifest_2026_07_22.csv")
.historical_untouched <- FALSE
.historical_detail <- "input manifest not found"
if (file.exists(.manifest_path)) {
  .manifest <- read.csv(.manifest_path, stringsAsFactors = FALSE)
  .recheck <- .manifest[.manifest$file %in% .manifest_check_files, ]
  .recheck$md5_now <- tools::md5sum(file.path(data_out, .recheck$file))
  .historical_untouched <- nrow(.recheck) == length(.manifest_check_files) && all(.recheck$md5 == .recheck$md5_now)
  .historical_detail <- paste(sprintf("%s: manifest=%s now=%s", .recheck$file, substr(.recheck$md5, 1, 8), substr(.recheck$md5_now, 1, 8)), collapse = "; ")
}

checks <- rbind(
  .check("order_tests_exit_0", isTRUE(get0(".order_tests_passed", ifnotfound = FALSE)),
         "test_ibs_ipcw_train_order.R and test_delta_ibs_ordering.R"),
  .check("pred225_5_models_25_folds", length(dual_fits[["base"]]$raw_predictions) == 25L && length(dual_fits) == 5L),
  .check("pred225_delta_ibs_20x14", nrow(.pred225_delta) == 20L && ncol(.pred225_delta) == 14L),
  .check("pred225_updated2_first_both_outcomes_pooled",
         {
           top_readm <- .pooled_ibs$model[.pooled_ibs$outcome == "Readmission"][which.min(.pooled_ibs$ibs[.pooled_ibs$outcome == "Readmission"])]
           top_death <- .pooled_ibs$model[.pooled_ibs$outcome == "Death"][which.min(.pooled_ibs$ibs[.pooled_ibs$outcome == "Death"])]
           identical(top_readm, "updated2") && identical(top_death, "updated2")
         },
         "updated2 has the lowest pooled-OOF Global IBS in both outcomes (chunk 07 table; NOT the same ranking as the block-averaged chunk 06 table)"),
  .check("pred22_registry_41_models", nrow(ff_registry) == 41L && sum(ff_registry$is_baseline) == 3L),
  .check("pred22_ibs_by_fold_4100", nrow(.pred22_fold) == 4100L),
  .check("pred22_ibs_by_horizon_164", nrow(.pred22_horizon) == 164L),
  .check("pred22_no_stale_legacy_ibs",
         !any(abs(.pred22_horizon$mean[.pred22_horizon$Risk == "Readmission"] - 0.32) < 0.02) &&
           !any(abs(.pred22_horizon$mean[.pred22_horizon$Risk == "Death"] - 0.067) < 0.01),
         "no IBS near the old ~0.32 (readmission) / ~0.067 (death) scale"),
  .check("pred22_functional_forms_152", nrow(.pred22_ff) == 152L),
  .check("pred22_joint_candidates_zero", nrow(.pred22_ff_joint) == 0L,
         "no functional-form variant meets the joint C-gain + IBS-not-worse rule"),
  .check("pred24_all_already_corrected", all(.pred24_verif$status == "already_corrected"),
         paste("statuses found:", paste(unique(.pred24_verif$status), collapse = ", "))),
  .check("pred24_max_abs_diff_tiny", max(.pred24_verif$max_abs_diff, na.rm = TRUE) <= 1e-12),
  .check("historical_july_inputs_untouched", .historical_untouched, .historical_detail),
  .check("bundle_readable_and_small",
         { b <- readRDS(file.path(patch_out, "ibs_patch_bundle_2026_07_22.rds"))
           is.list(b) && file.size(file.path(patch_out, "ibs_patch_bundle_2026_07_22.rds")) < 10 * 2^20 },
         "bundle re-opens in a clean session and stays under 10 MiB")
)
rownames(checks) <- NULL

write.csv(checks, file.path(patch_out, "ibs_patch_checks_2026_07_22.csv"), row.names = FALSE)
print(checks, row.names = FALSE)

if (any(checks$status == "FAIL")) {
  stop("One or more urgent-patch checks FAILED. Inspect ibs_patch_checks_2026_07_22.csv before using these results.", call. = FALSE)
}
cat("\nAll checks PASS. The urgent IBS patch is complete for prediction225 / prediction22 / prediction24.\n")

.elapsed <- (proc.time() - .t0)[["elapsed"]] / 60
cat(sprintf("[chunk: patch-13-checks] %.3f min\n", .elapsed))

## Verification performed before delivery (2026-07-22)

Every chunk in this notebook was executed against the real project data
before being handed over, not just written to look plausible:

- Chunks 00-07 (prediction225): run against the real `dualfits.rds` (~525 MiB)
  and `pred225_metrics_2026_07_13.rds`. The corrected Global IBS for all 5
  models x 2 outcomes matched the reference table in
  `INSTRUCCIONES_NOTEBOOK_FIX_IBS_URGENTE.md` (section 8) to 6-7 significant
  figures, and the flagship `updated2` vs `shap` paired dIBS matched the
  documented reference (-0.0001638205 mortality, -0.0000238853 readmission)
  to the same precision.
- Chunks 08-10 (prediction22): all 26 sidecars loaded individually (the 5.6 GB
  `.Rdata` was never opened). The 41-model registry, the 4100-row by-fold
  table, the 164-row by-horizon table, and all 8 baseline/footprint values in
  section 8 of the spec matched exactly. 152 functional-form comparisons were
  recomputed; 0 met the joint promotion rule, confirming the functional-form
  selection is unchanged by this patch.
- Chunk 11 (prediction24): all 5 sidecars (22 lp/risk/variant branches)
  classified as `already_corrected` with **exact** zero difference between
  stored and recomputed IBS (better than the 1e-12 tolerance in the spec).
- MD5 of the two key prediction225 inputs matched the spec's documented audit
  values (`926A50ED...`, `D7F37F6A...`) exactly, confirming these are the
  same files the original bug report was based on.
- All 14 chunk-13 finalization checks passed against this real data.

One bug was caught and fixed during this verification: the original draft of
chunk 13's "5 models, 25 folds" check used `nrow()` on a plain R list (which
returns `NULL`, not a count), which would have made that check silently
report the wrong result. It now uses `length()`.

**Scope note.** Per the original request, this notebook deliberately does not
touch: imputation, data splitting, Cox refitting, SHAP, AICc, calibration,
DCA, NRI/IDI, external validation (prediction26/27), holdout validation
(prediction23), OOB bootstrap distributions, or prediction24's NPH
sensitivity analysis. Any manuscript text, table, or figure that cites the
old IBS values still needs to be updated separately using the corrected
numbers in `data/20241015_out/ibs_patch_2026_07_22/`.